# Toll Plaza Queue Simulation

**Research question:** How many toll booths must be open at peak time to keep the average waiting time under two minutes?

Model: M/M/c queue. Cars arrive at random (rate λ per minute), each booth serves at rate µ per minute, and c booths are open.
Traffic intensity ρ = λ / (cµ). A steady state exists only when ρ < 1.

Notebook plan (run top to bottom with *Run all*):
1. Setup and parameters
2. Theory: M/M/1 and Erlang C formulas
3. Discrete-event simulator (random variates + event list)
4. Verification (assert) for M/M/1, then M/M/c
5. Experiment: average wait vs number of booths, with 95% confidence intervals
6. Answer

## 1. Setup and parameters

In [ ]:
import heapq
import math
from collections import deque

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# --- Peak-time model parameters (units: cars per minute) ---
LAMBDA = 6.0   # arrival rate: cars arriving per minute at peak
MU = 2.5       # service rate: cars ONE booth can serve per minute
TARGET = 2.0   # maximum acceptable average wait (minutes)

BASE_SEED = 20261124   # change this to get a different (but reproducible) set of runs

for c in range(1, 6):
    rho = LAMBDA / (c * MU)
    print(f"c = {c}: rho = {rho:.3f}  ->", "stable" if rho < 1 else "UNSTABLE (queue grows without bound)")

## 2. Theory (the exact answers we will check the simulation against)

**M/M/1** (one booth): mean wait in the queue

$$W_q = \frac{\rho}{\mu - \lambda}, \qquad \rho = \frac{\lambda}{\mu}$$

**M/M/c** (Erlang C): with offered load $a = \lambda/\mu$ and $\rho = a/c < 1$,

$$P_{wait} = \frac{\dfrac{a^c}{c!}\dfrac{1}{1-\rho}}{\displaystyle\sum_{k=0}^{c-1}\frac{a^k}{k!} + \frac{a^c}{c!}\frac{1}{1-\rho}}, \qquad W_q = \frac{P_{wait}}{c\mu - \lambda}$$

For c = 1 the Erlang C formula reduces to the M/M/1 formula, which is a useful sanity check in itself.

In [ ]:
def mm1_wq(lam, mu):
    """Exact mean waiting time in the queue for M/M/1 (minutes)."""
    if lam >= mu:
        raise ValueError("M/M/1 needs lambda < mu")
    rho = lam / mu
    return rho / (mu - lam)


def erlang_c_pwait(lam, mu, c):
    """Erlang C: probability that an arriving car has to wait."""
    a = lam / mu
    rho = a / c
    if rho >= 1:
        raise ValueError("Erlang C needs rho = lambda/(c*mu) < 1")
    top = (a ** c / math.factorial(c)) / (1 - rho)
    bottom = sum(a ** k / math.factorial(k) for k in range(c)) + top
    return top / bottom


def erlang_c_wq(lam, mu, c):
    """Exact mean waiting time in the queue for M/M/c (minutes)."""
    return erlang_c_pwait(lam, mu, c) / (c * mu - lam)


# Sanity check on the theory code itself: Erlang C with c = 1 must equal the M/M/1 formula
assert abs(erlang_c_wq(2.0, 2.5, 1) - mm1_wq(2.0, 2.5)) < 1e-12

print("Theoretical mean wait in queue at peak (lambda = %.1f, mu = %.1f):" % (LAMBDA, MU))
for c in range(1, 6):
    if LAMBDA / (c * MU) < 1:
        print(f"  c = {c}: Wq = {erlang_c_wq(LAMBDA, MU, c):.4f} min   (P(wait) = {erlang_c_pwait(LAMBDA, MU, c):.3f})")
    else:
        print(f"  c = {c}: no steady state (rho >= 1)")

## 3. The discrete-event simulator

Instead of stepping through time second by second, the program keeps an **event list** (a priority queue ordered by time) and jumps straight to the next event:

* **Arrival:** schedule the next arrival (random gap). If a booth is free, the car starts service at once (wait = 0) and a departure is scheduled. Otherwise it joins the FIFO queue.
* **Departure:** if cars are waiting, the first one starts service (its wait is now minus its arrival time) and a new departure is scheduled. Otherwise the booth becomes idle.

**Random variate generation:** gaps between arrivals and service times are exponential. We draw them from the generator `rng.exponential(1/rate)`, which turns uniform random numbers into exponential variates (inverse transform: $-\ln(U)/\text{rate}$).

In [ ]:
ARRIVAL, DEPARTURE = 1, 0   # on a time tie, process departures first


def simulate_mmc(lam, mu, c, n_customers, rng, warmup_frac=0.1):
    """
    One replication of an M/M/c queue (FIFO).

    Runs until `n_customers` cars have STARTED service, discards the first
    `warmup_frac` of them (start-up bias), and returns the mean waiting time
    in the queue (minutes) of the rest.
    """
    events = []                                   # event list: (time, kind)
    heapq.heappush(events, (rng.exponential(1.0 / lam), ARRIVAL))
    busy = 0                                      # booths currently serving
    queue = deque()                               # arrival times of waiting cars
    waits = []                                    # wait of each car, in order of service start

    while len(waits) < n_customers:
        t, kind = heapq.heappop(events)
        if kind == ARRIVAL:
            heapq.heappush(events, (t + rng.exponential(1.0 / lam), ARRIVAL))   # next arrival
            if busy < c:                          # a booth is free
                busy += 1
                waits.append(0.0)
                heapq.heappush(events, (t + rng.exponential(1.0 / mu), DEPARTURE))
            else:                                 # all booths busy: join the queue
                queue.append(t)
        else:                                     # DEPARTURE
            if queue:                             # next car in line starts service
                arrived_at = queue.popleft()
                waits.append(t - arrived_at)
                heapq.heappush(events, (t + rng.exponential(1.0 / mu), DEPARTURE))
            else:
                busy -= 1

    start = int(warmup_frac * len(waits))
    return float(np.mean(waits[start:]))


def replicate(lam, mu, c, n_reps, n_customers, seed, warmup_frac=0.1):
    """
    Monte Carlo: run `n_reps` independent replications (each with its own seed
    stream) and return (mean, half_width_95, array_of_replication_means).
    """
    streams = np.random.SeedSequence(seed).spawn(n_reps)
    means = np.array([
        simulate_mmc(lam, mu, c, n_customers, np.random.default_rng(s), warmup_frac)
        for s in streams
    ])
    mean = means.mean()
    half = stats.t.ppf(0.975, n_reps - 1) * means.std(ddof=1) / math.sqrt(n_reps)
    return mean, half, means

## 4. Verification

Before trusting any answer, the simulator must reproduce the exact theory. We use `assert` so the notebook **stops with an error** if the simulation and theory disagree.

Each check asserts that the simulated mean is within a small relative tolerance (5%) of the exact theoretical value. The 95% confidence interval is printed alongside so you can see that theory also falls within the simulation's uncertainty.

### 4a. One booth (M/M/1)

In [ ]:
TOL = 0.05   # accept up to 5% relative difference between simulation and theory

def verify(lam, mu, c, theory, n_reps=40, n_customers=60_000, seed=BASE_SEED):
    mean, half, _ = replicate(lam, mu, c, n_reps, n_customers, seed)
    rel_err = abs(mean - theory) / theory
    print(f"c={c}  lam={lam}, mu={mu}:  simulated = {mean:.4f} +/- {half:.4f}   theory = {theory:.4f}   rel. error = {rel_err:.2%}")
    assert rel_err < TOL, f"relative error {rel_err:.2%} exceeds tolerance {TOL:.0%}"
    return mean, half

# M/M/1: lambda = 2, mu = 2.5  (rho = 0.8)
verify(2.0, 2.5, 1, mm1_wq(2.0, 2.5))
print("M/M/1 verification PASSED")

### 4b. Several booths (M/M/c vs Erlang C)

In [ ]:
# Use the real peak-time parameters for c = 3, 4, 5, plus one extra case with a different lambda/mu
for c in (3, 4, 5):
    verify(LAMBDA, MU, c, erlang_c_wq(LAMBDA, MU, c))

verify(4.0, 1.0, 5, erlang_c_wq(4.0, 1.0, 5))   # rho = 0.8, a different shape of system
print("M/M/c verification (Erlang C) PASSED")

## 5. Experiment: average wait vs number of booths

* For **c = 1 and 2**, ρ ≥ 1, so there is no steady state. The simulated wait just keeps growing the longer we run. We show this by running for two different lengths.
* For **c = 3, 4, 5**, ρ < 1. We run 50 replications of 100,000 cars each and report the mean wait with a 95% confidence interval (CI).

In [ ]:
# --- Unstable cases: wait grows with run length ---
print("Unstable systems (rho >= 1): mean wait keeps growing with run length")
for c in (1, 2):
    short = replicate(LAMBDA, MU, c, 10, 5_000, BASE_SEED)[0]
    long_ = replicate(LAMBDA, MU, c, 10, 50_000, BASE_SEED)[0]
    print(f"  c = {c} (rho = {LAMBDA/(c*MU):.2f}):  5,000 cars -> {short:8.2f} min     50,000 cars -> {long_:9.2f} min")

In [ ]:
# --- Stable cases: mean wait with 95% CI ---
N_REPS, N_CUST = 50, 100_000
results = {}
for c in (3, 4, 5):
    mean, half, _ = replicate(LAMBDA, MU, c, N_REPS, N_CUST, BASE_SEED + c)
    results[c] = (mean, half, erlang_c_wq(LAMBDA, MU, c))

print(f"{'c':>2} {'rho':>6} {'sim mean (min)':>15} {'95% CI half-width':>18} {'Erlang C (min)':>15}")
for c, (m, h, th) in results.items():
    print(f"{c:>2} {LAMBDA/(c*MU):6.3f} {m:15.4f} {h:18.4f} {th:15.4f}")

In [ ]:
cs = list(results)
means = [results[c][0] for c in cs]
halves = [results[c][1] for c in cs]
theory = [results[c][2] for c in cs]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(cs, means, yerr=halves, fmt="o-", capsize=5, label="Simulation (mean, 95% CI)")
ax.plot(cs, theory, "s--", color="gray", label="Erlang C (exact)")
ax.axhline(TARGET, color="red", linestyle=":", label=f"Target = {TARGET} min")
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xlabel("Number of open booths, c")
ax.set_ylabel("Average wait in queue (minutes)")
ax.set_title(f"Toll plaza at peak: lambda = {LAMBDA}/min, mu = {MU}/min per booth")
ax.annotate("c = 1, 2: unstable\n(queue grows without bound)", xy=(1.5, TARGET * 0.5), ha="center", color="darkred")
ax.set_ylim(0, max(4, max(means) * 1.3))
ax.legend()
fig.tight_layout()
fig.savefig("wait_vs_booths.png", dpi=150)
plt.show()

## 6. The answer

In [ ]:
ok = [c for c in cs if results[c][0] + results[c][1] < TARGET]   # upper CI limit below target
c_min = min(ok)
m, h, th = results[c_min]

print("RESULT")
print("  With 1 or 2 booths, rho >= 1 and the queue grows without bound.")
for c in cs:
    mm, hh, _ = results[c]
    print(f"  With {c} booths: average wait = {mm:.2f} min (95% CI {mm-hh:.2f} to {mm+hh:.2f})")
print(f"  => At least {c_min} booths must be open at peak time to keep the average wait under {TARGET} minutes.")

assert LAMBDA / (c_min * MU) < 1
assert m + h < TARGET

---
# Extensions

The core project above is complete and verified. The three extensions below each end with an `assert` check.

## 7. Extension X3: rush hour (time-varying arrival rate)

At peak the arrival rate is not constant. Here a 3-hour morning is modelled as three phases:

| Time (min) | Phase | Arrival rate λ(t) |
|---|---|---|
| 0 – 60 | before rush | 3 cars/min |
| 60 – 120 | **rush hour** | 6 cars/min |
| 120 – 180 | after rush | 2 cars/min |

**Thinning (Lewis–Shedler):** generate candidate arrivals at the *maximum* rate λ_max = 6 and keep a candidate at time t with probability λ(t)/λ_max. The kept arrivals form a Poisson process with rate λ(t).

In [ ]:
T_END = 180.0
RATE_PROFILE = [(0.0, 60.0, 3.0), (60.0, 120.0, 6.0), (120.0, 180.0, 2.0)]   # (start, end, rate)
RUSH_START, RUSH_END = 60.0, 120.0
LAM_MAX = max(r for _, _, r in RATE_PROFILE)

def rate_tv(t):
    for a, b, r in RATE_PROFILE:
        if a <= t < b:
            return r
    return 0.0

def thinned_arrivals(rate_fn, lam_max, horizon, rng):
    """Arrival times on [0, horizon) of a Poisson process with rate rate_fn(t), by thinning."""
    t, out = 0.0, []
    while True:
        t += rng.exponential(1.0 / lam_max)              # candidate at the maximum rate
        if t >= horizon:
            return out
        if rng.random() < rate_fn(t) / lam_max:          # keep with probability rate(t)/lam_max
            out.append(t)

# ---- Check 1: the thinned arrivals have the right number of cars (expected = integral of lambda(t)) ----
expected_total = sum((b - a) * r for a, b, r in RATE_PROFILE)
expected_rush = (RUSH_END - RUSH_START) * 6.0
counts_total, counts_rush = [], []
for s in np.random.SeedSequence(BASE_SEED + 100).spawn(2000):
    arr = np.array(thinned_arrivals(rate_tv, LAM_MAX, T_END, np.random.default_rng(s)))
    counts_total.append(len(arr))
    counts_rush.append(np.sum((arr >= RUSH_START) & (arr < RUSH_END)))
print(f"cars per morning: simulated {np.mean(counts_total):.1f}  expected {expected_total:.1f}")
print(f"cars in rush hour: simulated {np.mean(counts_rush):.1f}  expected {expected_rush:.1f}")
print(f"variance/mean of total count: {np.var(counts_total)/np.mean(counts_total):.3f}   (Poisson => about 1)")
assert abs(np.mean(counts_total) - expected_total) / expected_total < 0.01
assert abs(np.mean(counts_rush) - expected_rush) / expected_rush < 0.01
assert 0.9 < np.var(counts_total) / np.mean(counts_total) < 1.1
print("X3 check 1 PASSED: thinning produces the right arrival counts")

In [ ]:
def simulate_tv(c, mu, arrivals, rng):
    """
    FIFO M(t)/M/c queue for a given list of arrival times (event-driven).
    Returns (arrival_times, waits) of every car, in order of arrival.
    """
    dep = []                      # heap of departure times of busy booths
    queue = deque()               # arrival times of waiting cars
    a_out, w_out = [], []
    i, n = 0, len(arrivals)
    INF = float("inf")
    while i < n or dep:
        next_arr = arrivals[i] if i < n else INF
        next_dep = dep[0] if dep else INF
        if next_dep <= next_arr:                       # departure (wins ties)
            t = heapq.heappop(dep)
            if queue:
                a = queue.popleft()
                a_out.append(a); w_out.append(t - a)
                heapq.heappush(dep, t + rng.exponential(1.0 / mu))
        else:                                          # arrival
            t = next_arr; i += 1
            if len(dep) < c:
                a_out.append(t); w_out.append(0.0)
                heapq.heappush(dep, t + rng.exponential(1.0 / mu))
            else:
                queue.append(t)
    return np.array(a_out), np.array(w_out)

# ---- Check 2: with a CONSTANT rate fed through thinning, the long-run mean wait must match Erlang C ----
def const_rate(t): return 6.0
reps = []
for s in np.random.SeedSequence(BASE_SEED + 200).spawn(10):
    rng = np.random.default_rng(s)
    arr = thinned_arrivals(const_rate, 10.0, 20_000.0, rng)       # lam_max = 10 > 6, so thinning really rejects
    a, w = simulate_tv(3, MU, arr, rng)
    reps.append(w[a > 2_000.0].mean())                            # discard warm-up
sim, th = np.mean(reps), erlang_c_wq(6.0, MU, 3)
print(f"constant rate through thinning, c=3: simulated {sim:.4f}  Erlang C {th:.4f}")
assert abs(sim - th) / th < 0.05
print("X3 check 2 PASSED: thinning + queue reproduces Erlang C")

In [ ]:
# ---- Experiment: wait during the rush-hour morning for different numbers of booths ----
N_REPS_TV = 200
BIN = 5.0
edges = np.arange(0, T_END + BIN, BIN)
tv = {}
for c in (2, 3, 4):
    overall, rush, binned = [], [], []
    for s in np.random.SeedSequence(BASE_SEED + 300 + c).spawn(N_REPS_TV):
        rng = np.random.default_rng(s)
        a, w = simulate_tv(c, MU, thinned_arrivals(rate_tv, LAM_MAX, T_END, rng), rng)
        overall.append(w.mean())
        rush.append(w[(a >= RUSH_START) & (a < RUSH_END)].mean())
        idx = np.digitize(a, edges) - 1
        binned.append([w[idx == k].mean() if np.any(idx == k) else np.nan for k in range(len(edges) - 1)])
    def ci(x):
        x = np.array(x); return x.mean(), stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / math.sqrt(len(x))
    tv[c] = dict(overall=ci(overall), rush=ci(rush), binned=np.nanmean(binned, axis=0))

print(f"{'c':>2} {'whole morning (min)':>22} {'rush hour only (min)':>24}")
for c, d in tv.items():
    print(f"{c:>2} {d['overall'][0]:12.3f} +/- {d['overall'][1]:.3f} {d['rush'][0]:15.3f} +/- {d['rush'][1]:.3f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
mid = edges[:-1] + BIN / 2
for c, d in tv.items():
    ax.plot(mid, d["binned"], marker="o", ms=3, label=f"{c} booths")
ax.axhline(TARGET, color="red", linestyle=":", label="Target = 2 min")
ax.axvspan(RUSH_START, RUSH_END, color="orange", alpha=0.15, label="Rush hour")
ax.set_xlabel("Time of arrival (minutes)"); ax.set_ylabel("Average wait (minutes)")
ax.set_title("X3: waiting time through a rush-hour morning"); ax.legend()
fig.tight_layout(); fig.savefig("x3_rush_hour.png", dpi=150); plt.show()

ok_tv = [c for c in tv if tv[c]["rush"][0] + tv[c]["rush"][1] < TARGET]
print(f"=> Fewest booths keeping the rush-hour average under {TARGET} min: {min(ok_tv)}")
assert min(ok_tv) >= 3, "2 booths cannot cope with a rush hour at rate 6 (capacity is only 5)"
assert tv[3]["rush"][0] < TARGET

## 8. Extension X2: second proof of correctness (error vs. number of samples)

The Monte Carlo error should shrink like $1/\sqrt{n}$ as the number of simulated cars $n$ grows. On a log–log plot this is a **straight line with slope −1/2**. If the simulator were biased or buggy, the error would level off instead of dropping.

For each $n$ we run many independent replications of an $n$-car simulation (c = 3) and measure the root-mean-square error against Erlang C.

In [ ]:
C_X2 = 3
theory_x2 = erlang_c_wq(LAMBDA, MU, C_X2)
ns = [1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000]
rmse = []
for n in ns:
    _, _, m = replicate(LAMBDA, MU, C_X2, 60, n, BASE_SEED + 400 + n)
    rmse.append(math.sqrt(np.mean((m - theory_x2) ** 2)))

slope, intercept = np.polyfit(np.log10(ns), np.log10(rmse), 1)
print(f"fitted log-log slope = {slope:.3f}   (theory: -0.5)")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.loglog(ns, rmse, "o-", label="Simulation RMSE vs Erlang C")
ref = rmse[0] * (np.array(ns) / ns[0]) ** -0.5
ax.loglog(ns, ref, "k--", label="slope -1/2 reference")
ax.set_xlabel("Cars simulated per run, n"); ax.set_ylabel("RMS error of mean wait (minutes)")
ax.set_title(f"X2: error vs sample size (fitted slope {slope:.2f})"); ax.legend(); ax.grid(True, which="both", alpha=0.3)
fig.tight_layout(); fig.savefig("x2_error_loglog.png", dpi=150); plt.show()

assert -0.65 < slope < -0.35, f"slope {slope:.3f} is not close to -0.5"
print("X2 check PASSED: error shrinks like 1/sqrt(n)")

## 9. Extension X5: which parameter matters more, λ or µ?

Change each parameter by ±5% (c = 3) and see how much the mean wait moves. The **elasticity** is (% change in wait) / (% change in parameter), computed by a central difference. A larger absolute elasticity means a bigger influence.

Theory predicts µ matters more: $W_q = \frac{1}{\mu} f(\lambda/\mu)$, so raising µ both lowers ρ *and* shortens every service. We check this with the simulation and the exact formula.

In [ ]:
DELTA = 0.05
C_X5 = 3
cases = {
    "baseline":   (LAMBDA, MU),
    "lambda +5%": (LAMBDA * (1 + DELTA), MU),
    "lambda -5%": (LAMBDA * (1 - DELTA), MU),
    "mu +5%":     (LAMBDA, MU * (1 + DELTA)),
    "mu -5%":     (LAMBDA, MU * (1 - DELTA)),
}
sim5, th5 = {}, {}
for name, (l, m) in cases.items():
    sim5[name] = replicate(l, m, C_X5, 50, 100_000, BASE_SEED + 500)[0]     # same seed for every case (common random numbers)
    th5[name] = erlang_c_wq(l, m, C_X5)
    print(f"{name:>11}: lambda={l:.3f} mu={m:.3f}  sim Wq = {sim5[name]:.4f}   Erlang C = {th5[name]:.4f}")

def elasticity(w, plus, minus):
    return ((w[plus] - w[minus]) / w["baseline"]) / (2 * DELTA)

E = {
    "lambda": (elasticity(sim5, "lambda +5%", "lambda -5%"), elasticity(th5, "lambda +5%", "lambda -5%")),
    "mu":     (elasticity(sim5, "mu +5%", "mu -5%"),         elasticity(th5, "mu +5%", "mu -5%")),
}
print()
for p, (es, et) in E.items():
    print(f"elasticity of Wq w.r.t. {p:>6}: simulation {es:+.2f}   exact {et:+.2f}")

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.bar(["lambda", "mu"], [abs(E['lambda'][0]), abs(E['mu'][0])], color=["tab:blue", "tab:green"])
ax.set_ylabel("|elasticity| of mean wait"); ax.set_title("X5: sensitivity of the wait (c = 3)")
fig.tight_layout(); fig.savefig("x5_sensitivity.png", dpi=150); plt.show()

# checks: simulation agrees with exact elasticities, and mu has the larger effect
assert abs(E["lambda"][0] - E["lambda"][1]) < 0.5 and abs(E["mu"][0] - E["mu"][1]) < 0.5
assert abs(E["mu"][1]) > abs(E["lambda"][1]) and abs(E["mu"][0]) > abs(E["lambda"][0])
print(f"X5 check PASSED: mu has the greater effect (|{E['mu'][0]:.2f}| vs |{E['lambda'][0]:.2f}|)")

In [ ]:
# Does the FINAL ANSWER (minimum booths) change if lambda or mu are a bit off?  (exact Erlang C)
def min_booths(l, m, target=TARGET):
    c = 1
    while l / (c * m) >= 1 or erlang_c_wq(l, m, c) >= target:
        c += 1
    return c

lam_mult = [0.8, 0.9, 1.0, 1.1, 1.2, 1.4, 1.6]
mu_mult = [0.8, 0.9, 1.0, 1.1, 1.2]
print("Minimum booths needed (rows: lambda multiplier, columns: mu multiplier)")
print("          " + "".join(f"mu x{m:<5}" for m in mu_mult))
for lm in lam_mult:
    print(f"lam x{lm:<4} " + "".join(f"{min_booths(LAMBDA*lm, MU*mm):>8} " for mm in mu_mult))
assert min_booths(LAMBDA, MU) == c_min     # same answer as the simulation experiment